# Qwen3.5 GDN 状态接口可行性探查

目的：确认 A100 环境、transformers 对 Qwen3.5 的支持，以及 **Gated DeltaNet (GDN) 循环状态**的可访问接口——
这是 TurboQuant vs LeapQuant 状态量化对比实验能否做的前提。用 Qwen3.5-0.8B（同架构、加载快）验证机制。

执行顺序按 cell 顺序；每步打印结论后再决定后续是否加载 9B 跑正式实验。


In [1]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
import torch
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
!pip -q install -U transformers accelerate bitsandbytes 2>&1 | tail -2
import transformers; print('transformers', transformers.__version__)


NVIDIA L4, 23034 MiB
torch 2.11.0+cu130 | cuda True NVIDIA L4
transformers 5.22.1


In [2]:
# 1) transformers 是否认识 Qwen3.5
import transformers.models as M
mods = [x for x in dir(M) if 'qwen' in x.lower()]
print('qwen modules in transformers:', mods)
from transformers import AutoConfig
for size in ['Qwen/Qwen3.5-0.8B', 'Qwen/Qwen3.5-9B']:
    try:
        cfg = AutoConfig.from_pretrained(size, trust_remote_code=True)
        print('\n==', size, '==', 'model_type=', getattr(cfg, 'model_type', None),
              '| arch=', getattr(cfg, 'architectures', None),
              '| layers=', getattr(cfg, 'num_hidden_layers', None))
        d = cfg.to_dict()
        keys = [k for k in d if any(s in k.lower() for s in ('delta','gate','linear','attn','state','rn','conv'))]
        print('  relevant cfg keys:', keys)
        for k in keys:
            print('   ', k, '=', str(d[k])[:160])
    except Exception as e:
        print('\n', size, 'config error:', type(e).__name__, e)


qwen modules in transformers: ['ColQwen2Config', 'ColQwen2ForRetrieval', 'ColQwen2PreTrainedModel', 'ColQwen2Processor', 'Qwen2AudioConfig', 'Qwen2AudioEncoder', 'Qwen2AudioEncoderConfig', 'Qwen2AudioForConditionalGeneration', 'Qwen2AudioModel', 'Qwen2AudioPreTrainedModel', 'Qwen2AudioProcessor', 'Qwen2Config', 'Qwen2ForCausalLM', 'Qwen2ForQuestionAnswering', 'Qwen2ForSequenceClassification', 'Qwen2ForTokenClassification', 'Qwen2Model', 'Qwen2MoeConfig', 'Qwen2MoeForCausalLM', 'Qwen2MoeForQuestionAnswering', 'Qwen2MoeForSequenceClassification', 'Qwen2MoeForTokenClassification', 'Qwen2MoeModel', 'Qwen2MoePreTrainedModel', 'Qwen2PreTrainedModel', 'Qwen2RMSNorm', 'Qwen2Tokenizer', 'Qwen2VLConfig', 'Qwen2VLForConditionalGeneration', 'Qwen2VLImageProcessor', 'Qwen2VLImageProcessorPil', 'Qwen2VLModel', 'Qwen2VLPreTrainedModel', 'Qwen2VLProcessor', 'Qwen2VLTextConfig', 'Qwen2VLTextModel', 'Qwen2VLVideoProcessor', 'Qwen2VLVisionConfig', 'Qwen2_5OmniAudioEncoderConfig', 'Qwen2_5OmniBigVGANConfi

config.json:   0%|          | 0.00/2.91k [00:00<?, ?B/s]


== Qwen/Qwen3.5-0.8B == model_type= qwen3_5 | arch= ['Qwen3_5ForConditionalGeneration'] | layers= None
  relevant cfg keys: ['output_hidden_states', 'return_dict']
    output_hidden_states = False
    return_dict = True


config.json:   0%|          | 0.00/3.13k [00:00<?, ?B/s]


== Qwen/Qwen3.5-9B == model_type= qwen3_5 | arch= ['Qwen3_5ForConditionalGeneration'] | layers= None
  relevant cfg keys: ['output_hidden_states', 'return_dict']
    output_hidden_states = False
    return_dict = True


In [3]:
# 2) 加载 0.8B，探查 GDN 循环状态接口
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
name = 'Qwen/Qwen3.5-0.8B'
tok = AutoTokenizer.from_pretrained(name, trust_remote_code=True)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32
model = AutoModelForCausalLM.from_pretrained(name, trust_remote_code=True,
                                             torch_dtype=dtype).to(device)
model.eval()
print('model class:', type(model).__module__, type(model).__name__)
# 打印含 delta/gate/state 的子模块
for n, m in model.named_modules():
    ln = n.lower()
    if any(s in ln for s in ('delta','gated','state','linear','conv','mamba','rnn')):
        print('MOD', n, '->', type(m).__name__)

tokenizer_config.json:   0%|          | 0.00/16.7k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 12.8MB            

tokenizer.json: downloading bytes:           |  0.00B            

chat_template.jinja:   0%|          | 0.00/7.75k [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/50.9k [00:00<?, ?B/s]

model.safetensors-00001-of-00001.safeten(…): reconstructing file:   0%|          |  0.00B / 1.75GB            

model.safetensors-00001-of-00001.safeten(…): downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

model class: transformers.models.qwen3_5.modeling_qwen3_5 Qwen3_5ForCausalLM
MOD model.layers.0.linear_attn -> Qwen3_5GatedDeltaNet
MOD model.layers.0.linear_attn.conv1d -> Conv1d
MOD model.layers.0.linear_attn.norm -> Qwen3_5RMSNormGated
MOD model.layers.0.linear_attn.out_proj -> Linear
MOD model.layers.0.linear_attn.in_proj_qkv -> Linear
MOD model.layers.0.linear_attn.in_proj_z -> Linear
MOD model.layers.0.linear_attn.in_proj_b -> Linear
MOD model.layers.0.linear_attn.in_proj_a -> Linear
MOD model.layers.1.linear_attn -> Qwen3_5GatedDeltaNet
MOD model.layers.1.linear_attn.conv1d -> Conv1d
MOD model.layers.1.linear_attn.norm -> Qwen3_5RMSNormGated
MOD model.layers.1.linear_attn.out_proj -> Linear
MOD model.layers.1.linear_attn.in_proj_qkv -> Linear
MOD model.layers.1.linear_attn.in_proj_z -> Linear
MOD model.layers.1.linear_attn.in_proj_b -> Linear
MOD model.layers.1.linear_attn.in_proj_a -> Linear
MOD model.layers.2.linear_attn -> Qwen3_5GatedDeltaNet
MOD model.layers.2.linear_attn.c

In [4]:
# 3) 一次前向 + 检查返回结构里是否携带状态（recurrent state / hidden_states / kv）
import torch
inp = tok('The capital of France is', return_tensors='pt').to('cuda')
with torch.no_grad():
    out = model(**inp, use_cache=True)
print('outputs keys:', [k for k in dir(out) if not k.startswith('_')])
print('has past_key_values:', hasattr(out, 'past_key_values'), type(out.past_key_values) if hasattr(out,'past_key_values') else None)
if hasattr(out, 'past_key_values') and out.past_key_values is not None:
    p = out.past_key_values
    print('past_key_values type:', type(p))
    try:
        # 逐层看元素形状
        for i, layer in enumerate(p):
            if isinstance(layer, (tuple, list)):
                print('layer', i, 'len', len(layer), 'shapes', [tuple(t.shape) for t in layer if hasattr(t,'shape')])
            else:
                print('layer', i, '=', type(layer))
    except Exception as e:
        print('iter pv err', e)
else:
    print('WARN: no past_key_values; Qwen3.5 GDN may store state inside model (need hook) or not support cache yet')


[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `chunk_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.


outputs keys: ['attentions', 'clear', 'copy', 'fromkeys', 'get', 'hidden_states', 'items', 'keys', 'logits', 'loss', 'move_to_end', 'past_key_values', 'pop', 'popitem', 'setdefault', 'to_tuple', 'update', 'values']
has past_key_values: True <class 'transformers.cache_utils.DynamicCache'>
past_key_values type: <class 'transformers.cache_utils.DynamicCache'>
iter pv err 'LinearAttentionLayer' object has no attribute 'keys'


In [5]:
# 4) 若存在 state 张量：定位其 dtype/shape，并确认与 delta rule S += w*v*k^T 一致
import torch
if hasattr(model, 'config'):
    print('model_type =', model.config.model_type)
print('GPU mem (GB):', round(torch.cuda.mem_get_info()[1]/2**30, 1))
print('\n== 探查结束 ==')
print('结论：请根据上面输出判断 —— (a) transformers 是否原生支持 Qwen3.5 GDN；')
print('      (b) 能否通过 past_key_values 拿到/替换每层状态（可做逐窗口/逐步状态量化）；')
print('      (c) 若不能，需要改用 flash-linear-attention(fla) 或自写 GDN forward。')


model_type = qwen3_5_text
GPU mem (GB): 22.0

== 探查结束 ==
结论：请根据上面输出判断 —— (a) transformers 是否原生支持 Qwen3.5 GDN；
      (b) 能否通过 past_key_values 拿到/替换每层状态（可做逐窗口/逐步状态量化）；
      (c) 若不能，需要改用 flash-linear-attention(fla) 或自写 GDN forward。
